In [9]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


STORAGE_OPTIONS = {
    'User-Agent': 'Our World In Data data fetch/1.0'
}


In [10]:
url_births_deaths = "https://ourworldindata.org/grapher/births-and-deaths-projected-to-2100.csv?v=1&csvType=full&useColumnShortNames=true"

try:
    df_bd = pd.read_csv(url_births_deaths, storage_options=STORAGE_OPTIONS)
except Exception as error:
    print("Fallo descarga remota, utilizando archivo local births-and-deaths-projected-to-2100.csv:", error)
    df_bd = pd.read_csv("../births-and-deaths-projected-to-2100.csv")

# Unificar datos históricos y proyecciones
df_bd["births"] = df_bd["births__sex_all__age_all__variant_estimates"].fillna(
    df_bd["births__sex_all__age_all__variant_medium__projected"]
)
df_bd["deaths"] = df_bd["deaths__sex_all__age_all__variant_estimates"].fillna(
    df_bd["deaths__sex_all__age_all__variant_medium__projected"]
)
df_bd = df_bd[["entity", "year", "births", "deaths"]]
df_bd.head(3)


,entity,year,births,deaths
0,Afghanistan,1950,383985.0,290972.0
1,Afghanistan,1951,391002.0,288752.0
2,Afghanistan,1952,397663.0,288059.0


In [11]:
url_pop = "https://ourworldindata.org/grapher/population-with-un-projections.csv?v=1&csvType=full&useColumnShortNames=true"
df_pop = pd.read_csv(url_pop, storage_options=STORAGE_OPTIONS)

df_pop["population"] = df_pop["population__sex_all__age_all__variant_estimates"].fillna(
    df_pop["population__sex_all__age_all__variant_medium__projected"]
)
df_pop = df_pop[["entity", "year", "population"]]
df_pop.head(3)


,entity,year,population
0,Afghanistan,1950,7776180.0
1,Afghanistan,1951,7879343.0
2,Afghanistan,1952,7987784.0


In [13]:
url_cm = "https://ourworldindata.org/grapher/child-mortality.csv?v=1&csvType=full&useColumnShortNames=true"
df_cm = pd.read_csv(url_cm, storage_options=STORAGE_OPTIONS)

df_cm = df_cm.rename(columns={"child_mortality_rate": "child_deaths_under_5"})
df_cm = df_cm[["entity", "year", "child_deaths_under_5"]]
df_cm.head(3)


,entity,year,child_deaths_under_5
0,Afghanistan,1957,36.82
1,Afghanistan,1958,36.21
2,Afghanistan,1959,35.64


In [14]:
df_clase = df_pop.merge(df_bd, on=["entity", "year"], how="inner")
df_clase = df_clase.merge(df_cm, on=["entity", "year"], how="inner")

print("Dimensiones del dataset de clase:", df_clase.shape)
print("Columnas:", df_clase.columns.tolist())
df_clase.head()


Dimensiones del dataset de clase: (13989, 6)
Columnas: ['entity', 'year', 'population', 'births', 'deaths', 'child_deaths_under_5']


,entity,year,population,births,deaths,child_deaths_under_5
0,Afghanistan,1957,8588340.0,432044.0,286114.0,36.82
1,Afghanistan,1958,8723412.0,439867.0,285911.0,36.21
2,Afghanistan,1959,8869270.0,447510.0,285500.0,35.64
3,Afghanistan,1960,9035048.0,456418.0,286159.0,35.06
4,Afghanistan,1961,9214082.0,465920.0,286818.0,34.52


In [15]:
url_health = "https://ourworldindata.org/grapher/total-healthcare-expenditure-gdp.csv?v=1&csvType=full&useColumnShortNames=true"
df_health = pd.read_csv(url_health, storage_options=STORAGE_OPTIONS)

df_health = df_health.rename(columns={
    "current_health_expenditure__che__as_percentage_of_gross_domestic_product__gdp__pct": "percentage_health"
})[["entity", "year", "percentage_health"]]

print("Dimensiones df_health:", df_health.shape)
df_health.head(3)


Dimensiones df_health: (4756, 3)


,entity,year,percentage_health
0,Afghanistan,2002,9.443391
1,Afghanistan,2003,8.941258
2,Afghanistan,2004,9.808474


In [16]:
url_gdp = "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true"
df_gdp = pd.read_csv(url_gdp, storage_options=STORAGE_OPTIONS)

df_gdp = df_gdp.rename(columns={
    "ny_gdp_pcap_pp_kd": "gdp"
})[["entity", "year", "gdp"]]

print("Dimensiones df_gdp:", df_gdp.shape)
df_gdp.head(3)


Dimensiones df_gdp: (7445, 3)


,entity,year,gdp
0,Afghanistan,2000,1617.8264
1,Afghanistan,2001,1454.1108
2,Afghanistan,2002,1774.3087


In [ ]:
url_le = "https://ourworldindata.org/grapher/life-expectancy-at-birth-including-the-un-projections.csv?v=1&csvType=full&useColumnShortNames=true"
df_le = pd.read_csv(url_le, storage_options=STORAGE_OPTIONS)

df_le["life_expectancy"] = df_le["life_expectancy__sex_all__age_0__variant_estimates"].fillna(
    df_le["life_expectancy__sex_all__age_0__variant_medium__projected"]
)
df_le = df_le[["entity", "year", "life_expectancy"]]

print("Dimensiones df_le:", df_le.shape)
df_le.head(3)

In [21]:
df_final = df_clase.merge(df_health, on=["entity", "year"], how="inner")
df_final = df_final.merge(df_gdp, on=["entity", "year"], how="inner")
df_final = df_final.merge(df_le, on=["entity", "year"], how="inner")

# Filtrar rango temporal 2000-2022
df_final = df_final[(df_final["year"] >= 2000) & (df_final["year"] <= 2022)]

columnas_requeridas = [
    'entity',
    'year',
    'population',
    'births',
    'deaths',
    'child_deaths_under_5',
    'percentage_health',
    'gdp',
    'life_expectancy'
]

df_final = df_final[columnas_requeridas]
df_final = df_final.sort_values(by=["entity", "year"]).reset_index(drop=True)

df_final.head(10)

,entity,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
0,Afghanistan,2002,21378123.0,1015202.0,239161.0,12.31,9.443391,1774.3087
1,Afghanistan,2003,22733053.0,1069727.0,243573.0,11.85,8.941258,1815.9282
2,Afghanistan,2004,23560656.0,1095372.0,244435.0,11.39,9.808474,1776.9182
3,Afghanistan,2005,24404575.0,1097038.0,241932.0,10.93,9.948289,1908.1147
4,Afghanistan,2006,25424100.0,1120880.0,245586.0,10.48,10.622766,1929.7239
5,Afghanistan,2007,25909852.0,1135656.0,245809.0,10.04,9.904675,2155.3530
6,Afghanistan,2008,26482631.0,1126525.0,235284.0,9.61,10.256495,2191.5044
7,Afghanistan,2009,27466102.0,1158017.0,237450.0,9.19,9.818487,2565.0220
8,Afghanistan,2010,28284088.0,1181113.0,237640.0,8.80,8.569672,2848.5862
9,Afghanistan,2011,29347708.0,1202456.0,235724.0,8.43,8.561908,2757.0525


In [18]:
df_final.info()


<class 'pandas.DataFrame'>
RangeIndex: 4430 entries, 0 to 4429
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   entity                4430 non-null   str    
 1   year                  4430 non-null   int64  
 2   population            4430 non-null   float64
 3   births                4430 non-null   float64
 4   deaths                4430 non-null   float64
 5   child_deaths_under_5  4430 non-null   float64
 6   percentage_health     4430 non-null   float64
 7   gdp                   4430 non-null   float64
dtypes: float64(6), int64(1), str(1)
memory usage: 277.0 KB


In [19]:
df_final.describe()


,year,population,births,deaths,child_deaths_under_5,percentage_health,gdp
count,4430.000000,4.430000e+03,4.430000e+03,4.430000e+03,4430.000000,4430.000000,4430.000000
mean,2011.575847,7.658912e+07,1.491242e+06,6.020810e+05,3.612691,6.229959,21880.117035
std,6.909472,5.427481e+08,1.047545e+07,4.244106e+06,3.973782,2.849768,23635.760850
min,2000.000000,9.566000e+03,1.930000e+02,6.600000e+01,0.130000,1.222602,711.976400
25%,2006.000000,1.923284e+06,3.155800e+04,1.421925e+04,0.840000,4.147871,4875.556500
50%,2012.000000,7.428585e+06,1.386450e+05,5.802700e+04,2.010000,5.687746,13332.022500
75%,2018.000000,2.634146e+07,6.160050e+05,2.026305e+05,5.080000,7.962407,30811.631500
max,2023.000000,8.091735e+09,1.460549e+08,6.972810e+07,48.930000,27.089685,149025.110000


In [20]:
ruta_guardado = "../data/processed/dataset_demografia_salud_gdp.csv"
os.makedirs(os.path.dirname(ruta_guardado), exist_ok=True)
df_final.to_csv(ruta_guardado, index=False)
print(f"Dataset guardado exitosamente en: {ruta_guardado}")

ruta_raw = "../data/raw/dataset_7variables_2000_2022.csv"
os.makedirs(os.path.dirname(ruta_raw), exist_ok=True)
df_final.to_csv(ruta_raw, index=False)
print(f"Dataset guardado exitosamente en: {ruta_raw}")


Dataset guardado exitosamente en: ../data/processed/dataset_demografia_salud_gdp.csv
